# Анализ качества VQA-аннотаций

**Модель:** Qwen2.5-VL-3B (через Ollama, Chat API)
**Датасет:** 10 000 мемов (Telegram + Bing)

Цель: оценить качество сгенерированных VQA-аннотаций перед построением поискового индекса.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

## 1. Загрузка данных

In [ ]:
records = []
with open('../data/processed/vqa_annotations.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

df = pd.DataFrame(records)
print(f'Всего записей: {len(df)}')
print(f'Столбцы: {df.columns.tolist()}')
df.head(3)

## 2. Обзор датасета

In [ ]:
# Классификация записей
df['has_caption'] = df['caption'].notna() & (df['caption'] != '')
df['has_all_fields'] = (
    df['has_caption'] & 
    df['tone'].notna() & 
    df['objects'].apply(lambda x: isinstance(x, list) and len(x) > 0) &
    df['main_idea'].notna()
)
df['is_raw'] = df.get('raw_response', pd.Series(dtype=str)).notna() & ~df['has_caption']

total = len(df)
full = df['has_all_fields'].sum()
cap_only = df['has_caption'].sum()
raw = df['is_raw'].sum()

print(f'=== Качество аннотаций ===')
print(f'Всего записей:              {total}')
print(f'✅ Полный JSON (4 поля):     {full} ({full*100//total}%)')
print(f'✅ С caption:                {cap_only} ({cap_only*100//total}%)')
print(f'⚠️  Обрезанный (raw_response): {raw} ({raw*100//total}%)')
print()

# По источникам
print('=== По источникам ===')
source_stats = df.groupby('source_type').agg(
    total=('filename', 'count'),
    with_caption=('has_caption', 'sum'),
    full_json=('has_all_fields', 'sum')
).reset_index()
source_stats['success_rate'] = (source_stats['with_caption'] / source_stats['total'] * 100).round(1)
display(source_stats)

In [ ]:
# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie: качество
labels = ['Полный JSON', 'Обрезанный']
sizes = [full, total - full]
colors = ['#2ecc71', '#f39c12']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('Качество JSON-аннотаций', fontsize=14)

# Bar: по источникам
src_counts = df['source_type'].value_counts()
bars = axes[1].bar(src_counts.index, src_counts.values, color=['#3498db', '#e74c3c'])
axes[1].set_title('Распределение по источникам', fontsize=14)
axes[1].set_ylabel('Количество')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                f'{int(bar.get_height())}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/vqa_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Длина описаний (Caption Length)

In [ ]:
df_ok = df[df['has_caption']].copy()
df_ok['caption_words'] = df_ok['caption'].apply(lambda x: len(str(x).split()))
df_ok['caption_chars'] = df_ok['caption'].apply(lambda x: len(str(x)))

print(f'=== Статистика caption ===')
print(f'Средняя длина: {df_ok["caption_words"].mean():.1f} слов ({df_ok["caption_chars"].mean():.0f} символов)')
print(f'Медиана:       {df_ok["caption_words"].median():.0f} слов')
print(f'Мин:           {df_ok["caption_words"].min()} слов')
print(f'Макс:          {df_ok["caption_words"].max()} слов')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# По словам
axes[0].hist(df_ok['caption_words'], bins=30, alpha=0.8, color='#3498db', edgecolor='white')
axes[0].axvline(df_ok['caption_words'].mean(), color='red', linestyle='--', label=f'mean={df_ok["caption_words"].mean():.1f}')
axes[0].set_xlabel('Количество слов')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение длины caption (слова)')
axes[0].legend()

# По источникам
for src in df_ok['source_type'].unique():
    subset = df_ok[df_ok['source_type'] == src]
    axes[1].hist(subset['caption_words'], bins=25, alpha=0.6, label=f'{src} (mean={subset["caption_words"].mean():.1f})')
axes[1].set_xlabel('Количество слов')
axes[1].set_ylabel('Частота')
axes[1].set_title('Длина caption по источникам')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/caption_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Распределение тонов (Tone)

In [ ]:
# Нормализуем тоны (некоторые модели пишут через /)
def normalize_tone(t):
    if pd.isna(t): return 'unknown'
    t = str(t).lower().strip()
    # Если модель вернула все варианты через /, берём первый
    if '/' in t and len(t) > 20:
        t = t.split('/')[0].strip()
    return t

df_ok['tone_norm'] = df_ok['tone'].apply(normalize_tone)
tone_counts = df_ok['tone_norm'].value_counts().head(10)

print('=== Топ-10 тонов ===')
for tone, count in tone_counts.items():
    print(f'  {tone:>25}: {count:>5} ({count*100//len(df_ok)}%)')

fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('husl', len(tone_counts))
bars = ax.barh(tone_counts.index[::-1], tone_counts.values[::-1], color=colors)
ax.set_xlabel('Количество')
ax.set_title('Распределение тонов мемов', fontsize=14)
for bar in bars:
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2.,
            f'{int(bar.get_width())}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/tone_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Количество объектов на мем

In [ ]:
def count_objects(objs):
    if isinstance(objs, list):
        return len(objs)
    return 0

df_ok['n_objects'] = df_ok['objects'].apply(count_objects)

print(f'=== Статистика объектов ===')
print(f'Среднее:  {df_ok["n_objects"].mean():.1f} объектов/мем')
print(f'Медиана:  {df_ok["n_objects"].median():.0f}')
print(f'Макс:     {df_ok["n_objects"].max()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_ok['n_objects'], bins=range(0, df_ok['n_objects'].max()+2),
             alpha=0.8, color='#9b59b6', edgecolor='white')
axes[0].set_xlabel('Количество объектов')
axes[0].set_ylabel('Частота')
axes[0].set_title('Объектов на мем')

# Самые частые объекты
all_objects = []
for objs in df_ok['objects']:
    if isinstance(objs, list):
        all_objects.extend([str(o).lower().strip() for o in objs])
obj_freq = Counter(all_objects).most_common(15)
obj_names, obj_counts = zip(*obj_freq) if obj_freq else ([], [])

axes[1].barh(list(obj_names)[::-1], list(obj_counts)[::-1], color='#e67e22')
axes[1].set_xlabel('Частота')
axes[1].set_title('Топ-15 объектов')

plt.tight_layout()
plt.savefig('../reports/objects_stats.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Примеры аннотаций

In [ ]:
pd.set_option('display.max_colwidth', 100)

# Лучшие примеры
examples = df_ok[df_ok['has_all_fields']].sample(10, random_state=42)
display(examples[['filename', 'source_type', 'caption', 'tone', 'objects']].reset_index(drop=True))

## 7. Проблемные записи

In [ ]:
df_raw = df[df['is_raw']].copy()
print(f'Обрезанных записей: {len(df_raw)}')

if len(df_raw) > 0:
    for i, (_, row) in enumerate(df_raw.head(5).iterrows()):
        raw = str(row.get('raw_response', ''))[:150]
        print(f'\n{i+1}. {row["filename"]} [{row["source_type"]}]')
        print(f'   Raw: {raw}...')

## 8. Word Cloud (частые слова в caption)

In [ ]:
import re

# Собираем все слова из caption
stop_words = {'a', 'an', 'the', 'is', 'in', 'on', 'of', 'and', 'to', 'with', 'its', 'it',
              'that', 'this', 'for', 'are', 'as', 'at', 'by', 'from', 'or', 'be', 'has', 'was'}
all_words = []
for cap in df_ok['caption'].dropna():
    words = re.findall(r'[a-zA-Z]+', str(cap).lower())
    all_words.extend([w for w in words if w not in stop_words and len(w) > 2])

word_freq = Counter(all_words).most_common(30)

fig, ax = plt.subplots(figsize=(14, 6))
words, counts = zip(*word_freq)
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(words)))
ax.barh(list(words)[::-1], list(counts)[::-1], color=colors)
ax.set_xlabel('Частота')
ax.set_title('Топ-30 слов в описаниях мемов', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Итоговая сводка

In [ ]:
print('=' * 50)
print('ИТОГОВАЯ СВОДКА')
print('=' * 50)
print(f'Модель:                Qwen2.5-VL-3B')
print(f'Датасет:               {total} мемов')
print(f'  - Telegram:          {len(df[df["source_type"]=="telegram"])}')
print(f'  - Bing:              {len(df[df["source_type"]=="bing"])}')
print(f'Success rate:          {cap_only*100/total:.1f}%')
print(f'Полный JSON:           {full*100/total:.1f}%')
print(f'Ср. длина caption:     {df_ok["caption_words"].mean():.1f} слов')
print(f'Ср. объектов/мем:      {df_ok["n_objects"].mean():.1f}')
print(f'Обрезанных:            {raw} ({raw*100/total:.1f}%)')
print('=' * 50)
print('\n✅ Данные готовы для построения эмбеддингов и поискового индекса.')